<a href="https://colab.research.google.com/github/yoshifumi0125/hellotalk-korean-phrases/blob/main/comfyUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. ComfyUI 本体とマネージャーをクローン
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git custom_nodes/ComfyUI-Manager

# 2. 依存ライブラリのインストール
!pip install -r requirements.txt
!pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121

# 3. Cloudflare トンネルをダウンロード（Mac からアクセスするための URL を発行）
!wget -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# 4. バックグラウンドでトンネルを起動
import subprocess
import threading
import time

def start_tunnel():
    subprocess.run(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8188", "--logfile", "tunnel.log"])

threading.Thread(target=start_tunnel, daemon=True).start()
time.sleep(3)

# 5. アクセス用 URL を表示して ComfyUI を起動
print("\n--- 以下の URL が表示されたらクリックして Mac のブラウザで開いてください ---")
!grep -o 'https://.*trycloudflare.com' tunnel.log || cat tunnel.log
print("----------------------------------------------------------------------\n")

!python main.py --listen 127.0.0.1 --port 8188

Cloning into 'ComfyUI'...
remote: Enumerating objects: 62152, done.
remote: Counting objects: 100% (399/399), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 62152 (delta 338), reused 251 (delta 249), pack-reused 61753 (from 4)
Receiving objects: 100% (62152/62152), 97.51 MiB | 54.74 MiB/s, done.
Resolving deltas: 100% (43035/43035), done.
/content/ComfyUI
Cloning into 'custom_nodes/ComfyUI-Manager'...
remote: Enumerating objects: 33452, done.
remote: Counting objects: 100% (428/428), done.
remote: Compressing objects: 100% (204/204), done.
remote: Total 33452 (delta 367), reused 232 (delta 224), pack-reused 33024 (from 3)
Receiving objects: 100% (33452/33452), 194.62 MiB | 79.15 MiB/s, done.
Resolving deltas: 100% (24765/24765), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 167.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 179.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 MB 142.4 MB/s  0:00:00


In [4]:
# 1. 必要なディレクトリを作成
!mkdir -p /content/ComfyUI/models/vae /content/ComfyUI/models/text_encoders /content/ComfyUI/models/diffusion_models

# 2. Video VAE のダウンロード（vae フォルダへ）
!wget -c https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_video_vae_int8_convrot.safetensors -P /content/ComfyUI/models/vae/

# 3. テキストエンコーダー Qwen3-VL 32B のダウンロード（text_encoders フォルダへ）
!wget -c https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors -P /content/ComfyUI/models/text_encoders/

# 4. メイン Diffusion モデルのダウンロード（diffusion_models フォルダへ）
!wget -c https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors -P /content/ComfyUI/models/diffusion_models/

# 2. Turbo モデルをダウンロード（loras フォルダへ）
!wget -c https://huggingface.co/lightx2v/Minimax-h3-Turbo/resolve/main/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors -P /content/ComfyUI/models/loras/
# （※ワークフローによって diffusion_models 側で読み込むノードもあるため、両方から参照できるようにリンクを作成）
!ln -sf /content/ComfyUI/models/loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors /content/ComfyUI/models/diffusion_models/


--2026-10-31 04:43:36--  https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_video_vae_int8_convrot.safetensors
Resolving huggingface.co (huggingface.co)... 3.167.112.25, 3.167.112.45, 3.167.112.96, ...
Connecting to huggingface.co (huggingface.co)|3.167.112.25|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/6a6bd1dc7034404148ec9107/702c89b26d42d714a57c498351f74db638b609fd059b5e590b7ce599e50d81c7?X-Xet-Cas-Uid=public&user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27minimax_h3_video_vae_int8_convrot.safetensors%3B+filename%3D%22minimax_h3_video_vae_int8_convrot.safetensors%22%3B&xip=aA0ulkUeo8o&Expires=1793425417&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNmE2YmQxZGM3MDM0NDA0MTQ4ZWM5MTA3LzcwMmM4OWIyNmQ0MmQ3MTRhNTdjNDk4MzUxZjc0ZGI2MzhiNjA5ZmQwNTliNWU1OTBiN2NlNTk5ZTUwZDgxYzdcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZ1c2VyX2lkPXB1Ymx

In [1]:
!ls -lh /content/ComfyUI/models/diffusion_models/ /content/ComfyUI/models/text_encoders/ /content/ComfyUI/models/vae/

/content/ComfyUI/models/diffusion_models/:
total 4.0K
lrwxrwxrwx 1 root root 87 Oct 31 04:45 minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors -> /content/ComfyUI/models/loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors
-rw-r--r-- 1 root root  0 Oct 31 04:10 put_diffusion_model_files_here

/content/ComfyUI/models/text_encoders/:
total 7.9G
-rw-r--r-- 1 root root    0 Oct 31 04:10 put_text_encoder_files_here
-rw-r--r-- 1 root root 7.9G Oct 31 04:45 qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors

/content/ComfyUI/models/vae/:
total 3.2G
-rw-r--r-- 1 root root 578M Oct 31 04:43 minimax_h3_audio_vae_fp32.safetensors
-rw-r--r-- 1 root root 2.7G Oct 31 04:44 minimax_h3_video_vae_int8_convrot.safetensors
-rw-r--r-- 1 root root    0 Oct 31 04:10 put_vae_here


In [2]:
%cd /content/ComfyUI
!python main.py --listen 127.0.0.1 --port 8188


/content/ComfyUI
[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[START] Security scan
[INFO] [ComfyUI-Manager] Using `uv` as Python module for pip operations.
Using Python 3.13.16 environment at: /usr
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-10-31 04:47:08.133
** Platform: Linux
** Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/ComfyUI
** ComfyUI Base Folder Path: /content/ComfyUI
** User directory: /content/ComfyUI/user
** ComfyUI-Manager config path: /content/ComfyUI/user/__manager/config.ini
** Log path: /content/ComfyUI/user/comfyui.log
Usi

In [5]:
# 1. ComfyUI フォルダへ移動
%cd /content/ComfyUI

# 2. 古いログを消去して新しいトンネルを起動
!rm -f tunnel.log
import subprocess
import threading
import time

def start_tunnel():
    subprocess.run(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8188", "--logfile", "tunnel.log"])

threading.Thread(target=start_tunnel, daemon=True).start()
time.sleep(5)

# 3. 新しく発行された URL を表示
!grep -o 'https://.*trycloudflare.com' tunnel.log

# 4. ComfyUI を起動
!python main.py --listen 127.0.0.1 --port 8188

/content/ComfyUI
https://continuing-modes-created-chair.trycloudflare.com
[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[START] Security scan
[INFO] [ComfyUI-Manager] Using `uv` as Python module for pip operations.
Using Python 3.13.16 environment at: /usr
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-10-31 05:07:13.992
** Platform: Linux
** Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/ComfyUI
** ComfyUI Base Folder Path: /content/ComfyUI
** User directory: /content/ComfyUI/user
** ComfyUI-Manager config path: /content/ComfyUI/user/__manager/conf

In [6]:
# 1. 16並列の超高速ダウンローダーを準備
!apt-get install -y aria2 > /dev/null

# 2. 保存先フォルダを作成
!mkdir -p /content/ComfyUI/models/diffusion_models

# 3. 19.5GB のメインモデルを爆速ダウンロード（数十秒〜1分程度）
!aria2c -x 16 -s 16 -k 1M -c "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors" -d /content/ComfyUI/models/diffusion_models/


10/31 05:12:53 [NOTICE] Downloading 1 item(s)

10/31 05:12:53 [NOTICE] CUID#7 - Redirecting to https://us.gcp.cdn.hf.co/xet-bridge-us/6a6bd1dc7034404148ec9107/c44113203bfee16cc28b9d344d14a306943708c3828f6a765962deb83e10651c?xip=aA0ulkUeo8o&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27minimax_h3_fl2va_pruned_int8_convrot.safetensors%3B+filename%3D%22minimax_h3_fl2va_pruned_int8_convrot.safetensors%22%3B&X-Xet-Cas-Uid=public&user_id=public&Expires=1793427173&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNmE2YmQxZGM3MDM0NDA0MTQ4ZWM5MTA3L2M0NDExMzIwM2JmZWUxNmNjMjhiOWQzNDRkMTRhMzA2OTQzNzA4YzM4MjhmNmE3NjU5NjJkZWI4M2UxMDY1MWNcXD94aXA9YUEwdWxrVWVvOG8mcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj0qJlgtWGV0LUNhcy1VaWQ9cHVibGljJnVzZXJfaWQ9cHVibGljIiwiQ29uZGl0aW9uIjp7IkRhdGVMZXNzVGhhbiI6eyJFcG9jaFRpbWUiOjE3OTM0MjcxNzN9fX1dfQ__&Signature=MEQCIBvOjTyeUegcQ3P7qv3eGjS-Zjvjbubanf6BUAfW5P4FAiAk1FY-TH3i%7Eehsj%7ESJGJf25wZ1REag4qvETgNte0u-7w__&Ke

In [9]:
%cd /content/ComfyUI

# 1. 古いログをリセットしてトンネルを起動
!rm -f tunnel.log
import subprocess
import threading
import time
import re

def start_tunnel():
    subprocess.run(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8188", "--logfile", "tunnel.log"])

threading.Thread(target=start_tunnel, daemon=True).start()

# 2. URL が発行されるまで待機して画面に大きく表示
url = None
for _ in range(15):
    time.sleep(1)
    try:
        with open("tunnel.log", "r") as f:
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
            if match:
                url = match.group(0)
                break
    except:
        pass

print("\n" + "="*60)
if url:
    print(f"👉 新しい ComfyUI の URL はこちらです:\n{url}")
else:
    print("URLの取得に少し時間がかかっています。Colabの左側ファイルから tunnel.log をご確認ください。")
print("="*60 + "\n")

# 3. ComfyUI を起動
!python main.py --listen 127.0.0.1 --port 8188

/content/ComfyUI

👉 新しい ComfyUI の URL はこちらです:
https://partners-establish-hunt-certain.trycloudflare.com

[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[START] Security scan
[INFO] [ComfyUI-Manager] Using `uv` as Python module for pip operations.
Using Python 3.13.16 environment at: /usr
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-10-31 05:30:23.449
** Platform: Linux
** Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/ComfyUI
** ComfyUI Base Folder Path: /content/ComfyUI
** User directory: /content/ComfyUI/user
** ComfyUI-Manager config path: /cont

In [10]:
!ls -lh /content/ComfyUI/models/diffusion_models/

total 20G
-rw-r--r-- 1 root root 20G Oct 31 05:13 c44113203bfee16cc28b9d344d14a306943708c3828f6a765962deb83e10651c
lrwxrwxrwx 1 root root  87 Oct 31 04:45 minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors -> /content/ComfyUI/models/loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors
-rw-r--r-- 1 root root   0 Oct 31 04:10 put_diffusion_model_files_here


In [11]:
!mv /content/ComfyUI/models/diffusion_models/c44113203bfee16cc28b9d344d14a306943708c3828f6a765962deb83e10651c /content/ComfyUI/models/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors

In [12]:
%cd /content/ComfyUI

# 1. 新しいトンネルを起動
!rm -f tunnel.log
import subprocess
import threading
import time
import re

def start_tunnel():
    subprocess.run(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8188", "--logfile", "tunnel.log"])

threading.Thread(target=start_tunnel, daemon=True).start()

# 2. 新しい URL を画面に表示
url = None
for _ in range(15):
    time.sleep(1)
    try:
        with open("tunnel.log", "r") as f:
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
            if match:
                url = match.group(0)
                break
    except:
        pass

print("\n" + "="*60)
if url:
    print(f"👉 新しい ComfyUI の URL はこちらです:\n{url}")
else:
    print("URLの取得に少し時間がかかっています。tunnel.log をご確認ください。")
print("="*60 + "\n")

# 3. ComfyUI を起動
!python main.py --listen 127.0.0.1 --port 8188

/content/ComfyUI

👉 新しい ComfyUI の URL はこちらです:
https://render-anything-cooper-cardiovascular.trycloudflare.com

[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[START] Security scan
[INFO] [ComfyUI-Manager] Using `uv` as Python module for pip operations.
Using Python 3.13.16 environment at: /usr
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-10-31 05:42:48.458
** Platform: Linux
** Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/ComfyUI
** ComfyUI Base Folder Path: /content/ComfyUI
** User directory: /content/ComfyUI/user
** ComfyUI-Manager config path:

In [13]:
# 15GB のテキストエンコーダーを確実に完全ダウンロード
!apt-get install -y aria2 > /dev/null
!aria2c -x 16 -s 16 -k 1M -c "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors" -d /content/ComfyUI/models/text_encoders/ -o qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors


10/31 05:55:54 [NOTICE] Downloading 1 item(s)

10/31 05:55:54 [NOTICE] CUID#7 - Redirecting to https://us.gcp.cdn.hf.co/xet-bridge-us/6a6bd1dc7034404148ec9107/e2674b88dea1ab1aa531a578128aec51fab6adb17920ef100e88931b455c328e?X-Xet-Cas-Uid=public&xip=aA0ulkUeo8o&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors%3B+filename%3D%22qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors%22%3B&user_id=public&Expires=1793429754&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNmE2YmQxZGM3MDM0NDA0MTQ4ZWM5MTA3L2UyNjc0Yjg4ZGVhMWFiMWFhNTMxYTU3ODEyOGFlYzUxZmFiNmFkYjE3OTIwZWYxMDBlODg5MzFiNDU1YzMyOGVcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZ4aXA9YUEwdWxrVWVvOG8mcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj0qJnVzZXJfaWQ9cHVibGljIiwiQ29uZGl0aW9uIjp7IkRhdGVMZXNzVGhhbiI6eyJFcG9jaFRpbWUiOjE3OTM0Mjk3NTR9fX1dfQ__&Signature=MEUCIQCrR5LAHxAfQBQIU1RrgT10ZWyd8Ykr4MG2QZfNU3sbugIgYrdK50V-GjPLD6rHNAkInT2bWUSJVdPnQQ13ovbeSTE_&Key-Pair-Id=01

In [8]:
# 1. 保存先フォルダを確実に用意
!mkdir -p /content/ComfyUI/models/diffusion_models

# 2. 16並列で 19.5GB を高速ダウンロード（進捗が % で表示されます）
!apt-get install -y aria2 > /dev/null
!aria2c -x 16 -s 16 -k 1M -c "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors" -d /content/ComfyUI/models/diffusion_models/


10/31 05:30:01 [NOTICE] Downloading 1 item(s)

10/31 05:30:02 [NOTICE] CUID#7 - Redirecting to https://us.gcp.cdn.hf.co/xet-bridge-us/6a6bd1dc7034404148ec9107/c44113203bfee16cc28b9d344d14a306943708c3828f6a765962deb83e10651c?user_id=public&xip=aA0ulkUeo8o&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27minimax_h3_fl2va_pruned_int8_convrot.safetensors%3B+filename%3D%22minimax_h3_fl2va_pruned_int8_convrot.safetensors%22%3B&X-Xet-Cas-Uid=public&Expires=1793428202&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNmE2YmQxZGM3MDM0NDA0MTQ4ZWM5MTA3L2M0NDExMzIwM2JmZWUxNmNjMjhiOWQzNDRkMTRhMzA2OTQzNzA4YzM4MjhmNmE3NjU5NjJkZWI4M2UxMDY1MWNcXD91c2VyX2lkPXB1YmxpYyZ4aXA9YUEwdWxrVWVvOG8mcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj0qJlgtWGV0LUNhcy1VaWQ9cHVibGljIiwiQ29uZGl0aW9uIjp7IkRhdGVMZXNzVGhhbiI6eyJFcG9jaFRpbWUiOjE3OTM0MjgyMDJ9fX1dfQ__&Signature=MEQCIEgsi18CE21hizBc6TP%7E3sexqMyHnQiJdAKuqWXFaTwWAiAlxVPjjC06qR7ypKtjO3jJt4zet8DgMdYfOKwlFSJXsA__&Key-

In [ ]:
%cd /content/ComfyUI

# 1. 新しいトンネルを起動
!rm -f tunnel.log
import subprocess
import threading
import time
import re

def start_tunnel():
    subprocess.run(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8188", "--logfile", "tunnel.log"])

threading.Thread(target=start_tunnel, daemon=True).start()

# 2. 新しい URL を画面に表示
url = None
for _ in range(15):
    time.sleep(1)
    try:
        with open("tunnel.log", "r") as f:
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
            if match:
                url = match.group(0)
                break
    except:
        pass

print("\n" + "="*60)
if url:
    print(f"👉 新しい ComfyUI の URL はこちらです:\n{url}")
else:
    print("URLの取得に少し時間がかかっています。tunnel.log をご確認ください。")
print("="*60 + "\n")

# 3. ComfyUI を起動
!python main.py --listen 127.0.0.1 --port 8188

/content/ComfyUI

👉 新しい ComfyUI の URL はこちらです:
https://texas-receiving-distributions-rose.trycloudflare.com

[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[START] Security scan
[INFO] [ComfyUI-Manager] Using `uv` as Python module for pip operations.
Using Python 3.13.16 environment at: /usr
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-10-31 05:57:16.469
** Platform: Linux
** Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/ComfyUI
** ComfyUI Base Folder Path: /content/ComfyUI
** User directory: /content/ComfyUI/user
** ComfyUI-Manager config path: /c